# Practical 05(b) — Lasso Regression

Study L1 regularization and its ability to select features by shrinking coefficients to zero.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score


## 2. Load multivariate dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(n=6000, random_state=42)
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]


## 3. Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 4. Fit baseline and Lasso models

In [ ]:
baseline = make_pipeline(StandardScaler(), LinearRegression())
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
rows = [{"Model": "Linear Regression", "Alpha": 0,
         "MSE": mean_squared_error(y_test, base_pred),
         "R2": r2_score(y_test, base_pred),
         "Non-zero coefficients": np.count_nonzero(baseline[-1].coef_)}]
models = {}
for alpha in [0.001, 0.01, 0.1, 1.0]:
    model = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=20000))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    models[alpha] = model
    rows.append({"Model": "Lasso", "Alpha": alpha,
                 "MSE": mean_squared_error(y_test, pred),
                 "R2": r2_score(y_test, pred),
                 "Non-zero coefficients": np.count_nonzero(model[-1].coef_)})
pd.DataFrame(rows)


## 5. Inspect selected features

In [ ]:
alpha = 0.1
selected = pd.DataFrame({"Feature": X.columns, "Coefficient": models[alpha][-1].coef_})
selected["Selected"] = selected["Coefficient"].abs() > 1e-8
selected.sort_values("Coefficient", key=abs, ascending=False)


## 6. Interpretation

Lasso applies an L1 penalty. Depending on alpha, it can set less useful feature coefficients exactly to zero and thereby perform feature selection.